# Handoff Claims Exception Routing

| Field | Value |
| --- | --- |
| Scenario id | `handoff-claims-exception-routing` |
| Pattern | `handoff` |
| API | `Responses API` |
| Recommended max tokens | `1000` per agent turn |

**Learning goal:** Learn how a triage agent grounds the routing decision in MCP facts, names the owner with a ROUTE directive, and how the routed specialist's decision always flows to a customer-communication finisher.

> Use Responses plus handoff orchestration when the correct owner of a claim exception depends on tool-grounded facts and every outcome must end with a customer communication.


### Runtime configuration

**Supporting code.** Reads the Ollama model and host from environment variables so the same notebook runs against any local setup without edits -- override `OLLAMA_MODEL` or `OLLAMA_HOST` before this cell to retarget it. It also creates `MCP_TOOL_FUNCTIONS`, the shared registry that fixture cells populate and `make_agent` later reads to grant tools by name. Nothing here touches the Agent Framework; this is the notebook's runtime dial.


In [ ]:
import os

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:12b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

# Domain tools register themselves here; every agent looks up its granted
# tools by name, so this registry is the one piece of shared runtime state.
MCP_TOOL_FUNCTIONS: dict[str, object] = {}

print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


### Notebook styling

**Supporting code.** Injects the Aptos-inspired CSS the rendering helpers rely on: roster cards, tool chips, and the per-agent accent bar that colors each transcript turn. `agent_color` hashes an agent's name to a stable palette entry, which is why the same agent keeps the same color across every cell and every run. Pure presentation -- no Agent Framework surface here.


In [ ]:
from IPython.display import HTML, Markdown, display


_AGENT_COLORS = ("#3868c8", "#b0530f", "#2f7d4f", "#7d3f98", "#a3374b", "#0f7d8a", "#8a6d0f", "#54596b")

_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
.maf-callout {
    border-left: 4px solid #3868c8; border-radius: 6px; padding: 0.6em 0.9em;
    margin: 0.6em 0; background: rgba(56, 104, 200, 0.08);
}
.maf-roster { display: flex; flex-wrap: wrap; gap: 0.6em; margin: 0.4em 0; }
.maf-card {
    border: 1px solid rgba(128, 128, 128, 0.35); border-radius: 8px;
    padding: 0.55em 0.8em; min-width: 14em; max-width: 24em; flex: 1;
}
.maf-card b { display: block; margin-bottom: 0.15em; }
.maf-card small { opacity: 0.75; }
.maf-chip {
    display: inline-block; border-radius: 999px; padding: 0 0.6em; margin: 0.2em 0.2em 0 0;
    font-size: 0.78em; border: 1px solid rgba(128, 128, 128, 0.4);
}
.maf-turn {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.45em 0.8em; margin: 0.45em 0; background: rgba(128, 128, 128, 0.07);
    white-space: pre-wrap;
}
.maf-turn b { color: var(--maf-agent, inherit); }
.maf-turn-label {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.3em 0.7em; margin: 0.7em 0 0.15em; background: rgba(128, 128, 128, 0.09);
}
.maf-turn-label b { color: var(--maf-agent, inherit); }
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


apply_notebook_style()


### Rendering helpers

**Supporting code.** `render_roster` draws one accent-colored card per agent listing its granted tools, and `render_transcript` splits workflow output on `[AgentName]` turn labels, rendering each turn's body as markdown beneath a colored label bar. This is what turns raw multi-agent output into the readable, color-coded conversation you see after the live run. Glue for the notebook, not framework API.


In [ ]:
import re as _re


def _escape_html(value) -> str:
    import html as _html

    return _html.escape(str(value))


def agent_color(name: str) -> str:
    """Deterministic per-agent accent color, stable across cells and runs."""

    return _AGENT_COLORS[sum(ord(ch) for ch in name) % len(_AGENT_COLORS)]


def render_callout(text: str) -> None:
    display(HTML("<div class='maf-callout'>" + _escape_html(text) + "</div>"))


def render_roster(scenario) -> None:
    """Render the agent roster as color-accented cards with tool chips."""

    cards = []
    for spec in scenario.agents:
        chips = "".join(
            "<span class='maf-chip'>" + _escape_html(tool) + "</span>" for tool in spec.mcp_tools
        ) or "<span class='maf-chip'>instructions only</span>"
        cards.append(
            "<div class='maf-card' style='border-top: 3px solid " + agent_color(spec.name) + "'>"
            + "<b>" + _escape_html(spec.name) + "</b>"
            + "<small>" + _escape_html(spec.description) + "</small>"
            + "<div>" + chips + "</div></div>"
        )
    display(HTML("<div class='maf-roster'>" + "".join(cards) + "</div>"))


_TURN_LABEL = _re.compile(r"^\[([A-Za-z0-9_]+)\]\s*", _re.MULTILINE)


def render_transcript(text: str) -> None:
    """Render workflow output as color-coded per-agent turns.

    Each turn's body is emitted as a ``text/markdown`` output (via
    ``Markdown``) so Jupyter renders the agent's markdown, while the
    per-agent accent color rides on an HTML label bar above the body.
    """

    pieces = _TURN_LABEL.split(text)
    preamble = pieces[0].strip()
    labeled = list(zip(pieces[1::2], pieces[2::2]))
    if not preamble and not labeled:
        display(Markdown(text))
        return
    if preamble:
        display(Markdown(preamble))
    for label, body in labeled:
        display(HTML(
            "<div class='maf-turn-label' style='--maf-agent: " + agent_color(label) + "'>"
            + "<b>" + _escape_html(label) + "</b></div>"
        ))
        display(Markdown(body.strip()))


## Pattern: Handoff

Handoff orchestration starts with triage, which names the owning specialist with a ROUTE directive. A code-defined router validates that choice against the allowed routes (falling back to keyword scoring), and scenarios with a designated finisher always end with that fixed owner completing the work.

Best fit: support, claims, entitlement, and exception flows where ownership depends on context.

## API Shape

**Responses API -- startup-selected scenario shape.** The scenario and orchestration pattern are wired in at server start. Each client request uses the standard OpenAI-compatible `/responses` body -- a plain chat-style input. The client never specifies which agents run; the server owns the orchestration entirely.

This is an enterprise scenario grounded by deterministic MCP context tools. In production those tools are served by a FastMCP stdio subprocess; this notebook inlines the same functions as plain callables so it runs without a local package or subprocess.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Triage names a ROUTE, the router validates it, one specialist runs (plus an optional fixed finisher). |
| Coordination | The router honors the triage ROUTE directive and falls back to keyword scoring. |
| Output | The output identifies the route, its source (directive or keywords), and the answers. |
| Best when | Use when the right owner depends on the request. |

## Instruction-Led LLM Agents

| Agent | Role | Capabilities |
| --- | --- | --- |
| `ClaimTriageAgent` | Classifies and routes the claim exception. | Domain tools: `lookup_enterprise_record`, `calculate_priority_score` |
| `PaymentSpecialistAgent` | Handles payment release exceptions. | Domain tools: `search_policy` |
| `FraudSpecialistAgent` | Investigates fraud signals. | Domain tools: `lookup_enterprise_record`, `search_policy` |
| `ComplianceSpecialistAgent` | Checks compliance holds. | Domain tools: `search_policy` |
| `CustomerCommsAgent` | Drafts the customer communication. | Domain tools: `create_decision_log_entry` |

> **Instructor note:** Each row is an LLM-backed agent with role instructions.
> Most agents rely on instructions alone; enterprise and quote-to-cash agents may
> also call domain tools for grounded context.


## How Handoff Plays Out in This Scenario

CLAIM-88120 exceeds the auto-approval threshold and carries a fraud signal, so per POL-CLM-09 triage must route fraud-first even though a payment specialist also has a claim on it. The router validates the ROUTE line, the fraud specialist investigates, and the communication agent always finishes with the customer message.

In this notebook `ClaimTriageAgent` triages and the router hands off to one of `PaymentSpecialistAgent`, `FraudSpecialistAgent`, `ComplianceSpecialistAgent`; `CustomerCommsAgent` always finishes.

## Pattern Comparison

| Pattern | Control flow | Coordination cost | Latency and cost | Fails when | Choose it when |
| --- | --- | --- | --- | --- | --- |
| sequential | Fixed pipeline; each stage consumes the previous stage's output | None at runtime -- the graph is the plan | Slowest wall-clock for independent work; easiest to debug and audit | A stage needs information only a later stage produces | The order is mandated: compliance gates, artifact chains, staged approvals |
| concurrent | One fan-out to parallel lanes, one code-defined fan-in | None between lanes; the fan-in labels and combines | Best wall-clock when lanes are truly independent | Lanes secretly depend on each other's findings | Independent expert reviews of the same input, under time pressure |
| **handoff (this notebook)** | Triage names one owner; a router validates the choice | One routing decision, code-checked against allowed routes | Cheapest -- only the owner (plus an optional finisher) runs | The case genuinely needs several specialists to collaborate | Ownership depends on case facts and most specialists should not run |
| group chat | Round-robin turns in a shared transcript; a closer ends each cycle | High -- every turn rereads the whole discussion | Slow and token-hungry; the transcript itself is the product | Participants never actually react to each other | Deliberation and critique must be visible in a recorded decision |
| magentic | A manager plans, delegates, observes a ledger, and replans | Highest -- planning, delegation, and replan loops | Most expensive and least predictable run shape | The task was really a known pipeline all along | Ambiguous work where the plan must change as evidence arrives |

> **Which pattern would we actually choose?** Handoff wins because policy makes ownership conditional on case facts -- the same claim without a fraud signal routes elsewhere -- and the finisher guarantees every path ends with a customer communication. Sequential would run specialists the claim does not need; group chat would debate what policy has already decided.


## MCP Tool Context

In production, these enterprise context functions are exposed by a local FastMCP stdio server and attached to
instruction-led LLM agents with `MCPStdioTool` using per-agent allowed tools. This notebook inlines
the same domain functions as plain callable tools so it remains standalone.


### Domain fixtures

**Supporting code.** The embedded enterprise records the tools read: vendors, alerts, claims, facilities, loan applications, acquisition targets, disputes, and the policy catalog and playbooks that govern them. Each record carries an engineered tension -- an expired review, a conflicting policy, a fraud signal tied with a merchant-error signal -- that this scenario's agents are supposed to surface. Inlined so the notebook stays self-contained; no MCP server, no Agent Framework surface.


In [ ]:
from typing import Any


_ENTERPRISE_RECORDS: dict[str, dict[str, Any]] = {
    "VENDOR-4471": {
        "type": "vendor",
        "name": "Northwind Analytics",
        "category": "data-platform",
        "annual_cost_usd": 184000,
        "data_classification": "confidential",
        "security_review": "expired",
        "owner": "Procurement",
        "notes": "Requested for the billing analytics rollout; SOC 2 report is 14 months old.",
    },
    "ALERT-2298": {
        "type": "security_alert",
        "name": "Anomalous OAuth token usage",
        "severity": "high",
        "affected_users": 3,
        "affected_endpoints": 2,
        "data_loss_indicators": False,
        "token_rotation_completed": False,
        "owner": "SecOps",
        "notes": "Three service accounts issued tokens from an unrecognized ASN within 9 minutes.",
    },
    "CLAIM-88120": {
        "type": "claim",
        "name": "Water damage exception",
        "amount_usd": 42150,
        "policy_id": "POLICY-PROP-12",
        "fraud_signals": 1,
        "compliance_holds": 0,
        "owner": "Claims",
        "notes": "Exceeds auto-approval threshold and includes one mismatched invoice date.",
    },
    "CLAIM-88121": {
        "type": "claim",
        "name": "Storm damage exception",
        "amount_usd": 58900,
        "policy_id": "POLICY-PROP-12",
        "fraud_signals": 2,
        "compliance_holds": 1,
        "owner": "Claims",
        "notes": "Duplicate invoice numbers plus an active regulatory hold; per POL-CLM-09 the fraud review precedes any payment decision.",
    },
    "POLICY-EX-77": {
        "type": "policy_exception",
        "name": "Temporary data residency waiver",
        "requested_by": "EU Sales",
        "risk_area": "data-residency",
        "duration_days": 90,
        "owner": "Governance",
        "notes": "Requests storing EU lead data in us-east during a vendor migration window.",
    },
    "FACILITY-DC-EAST": {
        "type": "facility",
        "name": "East Regional Data Center",
        "criticality": "tier-1",
        "dependent_services": ["billing", "auth", "exports"],
        "last_drill_days_ago": 410,
        "owner": "Operations",
        "notes": "Primary site for billing and auth; continuity drill is overdue.",
    },
    "FACILITY-DC-WEST": {
        "type": "facility",
        "name": "West Regional Data Center",
        "criticality": "tier-2",
        "dependent_services": ["reporting", "archive"],
        "last_drill_days_ago": 120,
        "owner": "Operations",
        "notes": "Secondary site with a current drill; a contrast case when prioritizing scope.",
    },
    "LOAN-73021": {
        "type": "loan_application",
        "name": "Home purchase mortgage application",
        "amount_usd": 384000,
        "credit_score": 764,
        "dti_ratio": 0.31,
        "ltv_ratio": 0.80,
        "employment_years": 6,
        "owner": "Lending",
        "notes": "Salaried applicant with two years of W-2s on file; a clean pass through every underwriting stage.",
    },
    "LOAN-73022": {
        "type": "loan_application",
        "name": "Home purchase mortgage application (marginal)",
        "amount_usd": 402000,
        "credit_score": 668,
        "dti_ratio": 0.44,
        "ltv_ratio": 0.92,
        "employment_years": 1,
        "owner": "Lending",
        "notes": "Self-employed applicant; DTI and LTV both exceed the POL-LEND-01 referral limits, so manual underwriting and compensating factors are required.",
    },
    "TARGET-ACQ-STELLAR": {
        "type": "acquisition_target",
        "name": "Stellar Metrics Ltd",
        "sector": "SaaS product analytics",
        "arr_usd": 24000000,
        "arr_growth_pct": 38,
        "logo_churn_pct": 9,
        "top_customer_revenue_share": 0.34,
        "open_litigation": 1,
        "soc2_status": "none",
        "owner": "Corporate Development",
        "notes": "Fast grower with one pending patent dispute, no SOC 2, a single-region deployment, and a third of revenue from one customer.",
    },
    "TARGET-ACQ-HARBOR": {
        "type": "acquisition_target",
        "name": "Harbor Data GmbH",
        "sector": "EU data-residency analytics",
        "arr_usd": 11000000,
        "arr_growth_pct": 12,
        "logo_churn_pct": 4,
        "top_customer_revenue_share": 0.11,
        "open_litigation": 0,
        "soc2_status": "current",
        "owner": "Corporate Development",
        "notes": "Slower but clean: current certifications, diversified revenue, and an EU footprint; the contrast case for the diligence lanes.",
    },
    "DISPUTE-90455": {
        "type": "transaction_dispute",
        "name": "Duplicate charge with lost-card report",
        "amount_usd": 1249.99,
        "merchant": "TechnoMart Online",
        "duplicate_posting": True,
        "card_reported_lost": True,
        "cardholder_present": False,
        "days_since_posting": 3,
        "owner": "Card Services",
        "notes": "The same amount posted twice (a merchant-error signal) and the cardholder reported the card lost the same week (a fraud signal); POL-DSP-04 makes fraud review win that tie.",
    },
    "DISPUTE-90456": {
        "type": "transaction_dispute",
        "name": "Subscription billed after cancellation",
        "amount_usd": 29.99,
        "merchant": "StreamBox Media",
        "duplicate_posting": False,
        "card_reported_lost": False,
        "cancellation_confirmed": True,
        "days_since_posting": 12,
        "owner": "Card Services",
        "notes": "A recurring charge posted twelve days after a confirmed cancellation; a clean subscription dispute with no fraud signal.",
    },
    "ADR-2209": {
        "type": "architecture_decision",
        "name": "Customer notification service: build versus buy",
        "annual_buy_cost_usd": 96000,
        "build_estimate_eng_quarters": 2,
        "vendor_sla_pct": 99.9,
        "vendor_data_region": "us-only",
        "platform_team_utilization_pct": 85,
        "owner": "Architecture",
        "notes": "The vendor processes data in the US only while a quarter of customers are in the EU; the build option lands on a platform team already at 85% utilization.",
    },
    "METRIC-CHURN-Q3": {
        "type": "metric_anomaly",
        "name": "Q3 enterprise churn spike",
        "baseline_monthly_churn_pct": 1.8,
        "current_monthly_churn_pct": 4.1,
        "spike_start": "week of Sep 8",
        "support_ticket_trend": "flat",
        "nps_delta": -12,
        "owner": "Customer Success",
        "notes": "Churn more than doubled while support volume stayed flat; the spike overlaps a Sep 1 pricing change and billing migration wave 2, so the cause is genuinely ambiguous.",
    },
    "SEGMENT-ENT-EU": {
        "type": "customer_segment",
        "name": "Enterprise EU segment",
        "account_count": 214,
        "arr_usd": 18600000,
        "renewal_concentration": "Q4-heavy",
        "recent_events": [
            "billing migration wave 2 (Sep 5-12)",
            "new DPA requirement emails",
            "P1 outages on Aug 28 and Sep 9",
        ],
        "owner": "Customer Success",
        "notes": "The segment where the churn spike concentrates; three overlapping candidate causes give an investigation real material to eliminate.",
    },
}

_POLICY_CATALOG: tuple[dict[str, Any], ...] = (
    {
        "id": "POL-PROC-01",
        "title": "Vendor Security Review",
        "summary": "Vendors handling confidential data require a security review no older than 12 months before purchase.",
        "keywords": ("vendor", "security", "procurement", "soc2", "review", "purchase"),
    },
    {
        "id": "POL-PROC-02",
        "title": "Spend Authorization Thresholds",
        "summary": "Spend above 100k USD requires budget owner plus finance director approval.",
        "keywords": ("budget", "spend", "procurement", "approval", "finance", "threshold"),
    },
    {
        "id": "POL-PROC-03",
        "title": "Regional Processing Exception",
        "summary": "Vendors may process confidential data in-region for up to 30 days during a migration window with security sign-off, even while the annual review is pending.",
        "keywords": ("vendor", "regional", "migration", "exception", "security", "processing"),
    },
    {
        "id": "POL-SEC-04",
        "title": "Identity Compromise Response",
        "summary": "Suspected token or identity compromise requires credential rotation and session revocation within one hour.",
        "keywords": ("identity", "token", "oauth", "security", "incident", "rotation"),
    },
    {
        "id": "POL-CLM-09",
        "title": "Claim Exception Routing",
        "summary": "Claims above the auto-approval threshold or with any fraud signal route to a specialist before payment.",
        "keywords": ("claim", "exception", "fraud", "payment", "threshold"),
    },
    {
        "id": "POL-GOV-03",
        "title": "Policy Exception Board",
        "summary": "Risk waivers require a documented business need, a compensating control, and a fixed expiry. Maximum waiver duration is 60 days.",
        "keywords": ("policy", "exception", "waiver", "risk", "compliance", "governance", "residency"),
    },
    {
        "id": "POL-BCP-02",
        "title": "Business Continuity Drills",
        "summary": "Tier-1 facilities must complete a continuity drill at least every 365 days.",
        "keywords": ("continuity", "drill", "facility", "tier-1", "operations", "recovery"),
    },
    {
        "id": "POL-LEND-01",
        "title": "Manual Underwriting Referral",
        "summary": "Loan applications with a debt-to-income ratio above 0.43 or a loan-to-value ratio above 0.90 require senior underwriter review and documented compensating factors before pricing.",
        "keywords": ("loan", "underwriting", "dti", "ltv", "credit", "referral", "lending"),
    },
    {
        "id": "POL-MA-02",
        "title": "Due Diligence Gate",
        "summary": "Acquisition recommendations require findings from the finance, legal, technology, and market workstreams; any single red flag blocks a proceed recommendation until a documented mitigation exists.",
        "keywords": ("acquisition", "diligence", "merger", "workstream", "target", "gate"),
    },
    {
        "id": "POL-DSP-04",
        "title": "Dispute Routing and Provisional Credit",
        "summary": "Disputes with any fraud indicator route to fraud review before provisional credit; pure merchant errors receive provisional credit within two business days; every dispute is acknowledged within ten business days.",
        "keywords": ("dispute", "fraud", "chargeback", "credit", "merchant", "routing", "card"),
    },
    {
        "id": "POL-ARCH-07",
        "title": "Build-versus-Buy Review",
        "summary": "Build-versus-buy decisions above 50k USD annual impact require a decision record covering total cost of ownership, security posture, data residency, and an exit strategy.",
        "keywords": ("architecture", "build", "buy", "vendor", "decision", "residency", "review"),
    },
)

_PLAYBOOKS: dict[str, list[str]] = {
    "procurement-approval": [
        "Confirm the request scope and the requesting business owner.",
        "Validate budget authority against the spend threshold policy.",
        "Confirm the vendor security review is current.",
        "Capture legal and data-protection terms that must be in the contract.",
        "Assemble the approval packet with a clear recommendation.",
    ],
    "security-enrichment": [
        "Pull the alert record and confirm severity.",
        "Enrich the identity dimension.",
        "Enrich the endpoint dimension.",
        "Enrich the network dimension.",
        "Assess data-loss indicators and assemble the incident summary.",
    ],
    "claims-exception": [
        "Normalize the claim and identify why it is an exception.",
        "Check the amount against the auto-approval threshold.",
        "Evaluate fraud signals and compliance holds.",
        "Route to the correct specialist.",
        "Draft the customer communication.",
    ],
    "policy-exception-board": [
        "State the requested exception and the affected policy.",
        "Assess the introduced risk.",
        "Document the business need and urgency.",
        "Define a compensating control and expiry date.",
        "Record the board recommendation.",
    ],
    "continuity-drill": [
        "Confirm the facility, criticality, and dependent services.",
        "Plan the drill scope and participants.",
        "Define IT failover and recovery objectives.",
        "Define communications and stakeholder updates.",
        "Define finance and operations contingencies.",
    ],
    "loan-origination": [
        "Normalize the application and confirm required documents.",
        "Pull credit and flag scores below the referral threshold.",
        "Verify income and recompute the debt-to-income ratio.",
        "Price the risk tier or refer for manual underwriting per policy.",
        "Assemble the offer packet with conditions and disclosures.",
    ],
    "due-diligence": [
        "Confirm the target profile and the deal thesis.",
        "Run finance, legal, technology, and market workstreams in parallel.",
        "Collect red flags and quantify each one.",
        "Check every red flag for a documented mitigation.",
        "Synthesize a proceed, renegotiate, or walk-away recommendation.",
    ],
    "dispute-resolution": [
        "Capture the dispute, the transaction, and the customer's account of events.",
        "Screen for fraud indicators before any credit decision.",
        "Route to the owning specialist based on the dominant signal.",
        "Resolve per policy and record the provisional credit decision.",
        "Send the customer the outcome and the regulatory-clock status.",
    ],
    "architecture-review": [
        "State the decision, the options, and the deadline.",
        "Score total cost of ownership for each option.",
        "Assess security posture and data residency for each option.",
        "Document the exit strategy for the preferred option.",
        "Record the board's decision with dissents and conditions.",
    ],
    "churn-investigation": [
        "Quantify the anomaly against baseline and segment it.",
        "List candidate causes with their supporting evidence.",
        "Assign specialists to confirm or eliminate each candidate.",
        "Reconcile findings and identify the dominant driver.",
        "Recommend remediation and an early-warning metric.",
    ],
}

_PRIORITY_TIERS: tuple[tuple[int, str], ...] = (
    (80, "critical"),
    (60, "high"),
    (40, "medium"),
    (0, "low"),
)



# Fixture data only -- the tools in the next cell read from these embedded records.
print("records:  ", ", ".join(sorted(_ENTERPRISE_RECORDS)))
print("policies: ", ", ".join(policy["id"] for policy in _POLICY_CATALOG))
print("playbooks:", ", ".join(sorted(_PLAYBOOKS)))


### Domain tools

**Supporting code.** Plain callables over the fixtures, registered in `MCP_TOOL_FUNCTIONS` so `make_agent` can grant them by name: record lookups, keyword policy search, a deterministic priority score, playbook steps, and a write-shaped decision log that never persists. In production these would be a FastMCP stdio server attached via `MCPStdioTool`; inlining preserves the exact tool contract while keeping the notebook self-contained. The cell ends with this scenario's grounding call so you see a real tool result before any agent runs.


In [ ]:
import hashlib
import json
from typing import Any


def _clamp(value: Any, low: int = 1, high: int = 5) -> int:
    try:
        number = int(value)
    except (TypeError, ValueError):
        number = low
    return max(low, min(high, number))


def lookup_enterprise_record(record_id: str) -> dict[str, Any]:
    """Look up a single embedded enterprise record by id."""

    key = (record_id or "").strip().upper()
    record = _ENTERPRISE_RECORDS.get(key)
    if record is None:
        return {"found": False, "record_id": record_id, "known_ids": sorted(_ENTERPRISE_RECORDS)}
    return {"found": True, "record_id": key, **record}


def search_policy(query: str) -> dict[str, Any]:
    """Search the embedded policy catalog with a simple keyword match."""

    terms = [term for term in (query or "").lower().replace(",", " ").split() if term]
    scored: list[tuple[int, dict[str, Any]]] = []
    for policy in _POLICY_CATALOG:
        haystack = " ".join((policy["title"], policy["summary"], " ".join(policy["keywords"]))).lower()
        score = sum(1 for term in terms if term in haystack)
        if score:
            scored.append((score, policy))
    scored.sort(key=lambda item: (-item[0], item[1]["id"]))
    matches = [
        {"id": policy["id"], "title": policy["title"], "summary": policy["summary"], "match_score": score}
        for score, policy in scored
    ]
    return {"query": query, "match_count": len(matches), "matches": matches}


def calculate_priority_score(impact: int, urgency: int, scope: int = 1) -> dict[str, Any]:
    """Compute a deterministic 0-100 priority score and tier."""

    impact_v = _clamp(impact)
    urgency_v = _clamp(urgency)
    scope_v = _clamp(scope)
    raw = (impact_v * 8) + (urgency_v * 8) + (scope_v * 4)
    tier = next(name for floor, name in _PRIORITY_TIERS if raw >= floor)
    return {"impact": impact_v, "urgency": urgency_v, "scope": scope_v, "score": raw, "tier": tier}


def list_playbook_steps(playbook: str) -> dict[str, Any]:
    """Return the ordered steps for an embedded playbook by name."""

    key = (playbook or "").strip().lower().replace("_", "-")
    steps = _PLAYBOOKS.get(key)
    if steps is None:
        return {"found": False, "playbook": playbook, "known_playbooks": sorted(_PLAYBOOKS)}
    return {
        "found": True,
        "playbook": key,
        "step_count": len(steps),
        "steps": [{"order": index, "action": action} for index, action in enumerate(steps, start=1)],
    }


def create_decision_log_entry(
    subject: str,
    decision: str,
    rationale: str = "",
    owner: str = "unassigned",
) -> dict[str, Any]:
    """Return the decision log entry that would be recorded."""

    fingerprint = "|".join((subject or "", decision or "", rationale or "", owner or ""))
    digest = hashlib.sha256(fingerprint.encode("utf-8")).hexdigest()[:12]
    return {
        "persisted": False,
        "entry_id": f"DLOG-{digest}",
        "subject": subject,
        "decision": decision,
        "rationale": rationale,
        "owner": owner,
    }


MCP_TOOL_FUNCTIONS.update(
    {
        "lookup_enterprise_record": lookup_enterprise_record,
        "search_policy": search_policy,
        "calculate_priority_score": calculate_priority_score,
        "list_playbook_steps": list_playbook_steps,
        "create_decision_log_entry": create_decision_log_entry,
    }
)

# Demo (offline): call one grounded tool directly before any agent exists.
print(json.dumps(lookup_enterprise_record("CLAIM-88120"), indent=2))


### Scenario schema

**Supporting code.** Plain frozen dataclasses -- `AgentSpec` and `ScenarioSpec` -- that mirror the embedded scenario JSON: identity, pattern, roster, token budget, and the pattern- specific fields (`handoff_finisher`, `concurrent_synthesizer`, `termination_phrases`). They are deliberately not framework types: keeping the scenario contract in plain data is what lets the same spec drive five different orchestrations and both API shapes.


In [ ]:
from dataclasses import dataclass
from typing import Any, Sequence


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    route_keywords: tuple[str, ...] = ()
    a2a_url: str | None = None


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_input: str
    agents: tuple[AgentSpec, ...]
    max_tokens: int
    handoff_finisher: str | None = None
    concurrent_synthesizer: str | None = None
    termination_phrases: tuple[str, ...] = ()


### Load the scenario

**Supporting code.** Hydrates the embedded JSON into the `SCENARIO` object every later cell reads -- the roster the agent factory builds from, the sample prompt the live run sends, and the budget the config uses. Change a field here (an instruction, a route keyword, the budget) and rerun the downstream cells to see how behavior shifts. Data plumbing, no Agent Framework surface.


In [ ]:
import json


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "handoff-claims-exception-routing",
  "pattern": "handoff",
  "title": "Handoff Claims Exception Routing",
  "learning_goal": "Learn how a triage agent grounds the routing decision in MCP facts, names the owner with a ROUTE directive, and how the routed specialist's decision always flows to a customer-communication finisher.",
  "when_to_use": "Use Responses plus handoff orchestration when the correct owner of a claim exception depends on tool-grounded facts and every outcome must end with a customer communication.",
  "max_tokens": 1000,
  "sample_input": "Route claim exception CLAIM-88120 (water damage, exceeds auto-approval) to the correct specialist and explain the decision.",
  "handoff_finisher": "CustomerCommsAgent",
  "concurrent_synthesizer": null,
  "termination_phrases": [],
  "agents": [
    {
      "name": "ClaimTriageAgent",
      "description": "Classifies and routes the claim exception.",
      "instructions": "Decide which specialist should own the claim: PaymentSpecialistAgent, FraudSpecialistAgent, or ComplianceSpecialistAgent. Use lookup_enterprise_record for the claim and calculate_priority_score to rank it. Per policy POL-CLM-09, any fraud signal routes to FraudSpecialistAgent before payment is considered; a compliance hold routes to ComplianceSpecialistAgent. End your reply with a line 'ROUTE: <AgentName>'.",
      "mcp_tools": [
        "lookup_enterprise_record",
        "calculate_priority_score"
      ],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "PaymentSpecialistAgent",
      "description": "Handles payment release exceptions.",
      "instructions": "Decide whether and how to release payment. Use search_policy to ground the auto-approval threshold and exception handling.",
      "mcp_tools": [
        "search_policy"
      ],
      "mcp_server": "enterprise_context",
      "route_keywords": [
        "payment",
        "release",
        "threshold",
        "approval"
      ],
      "a2a_url": null
    },
    {
      "name": "FraudSpecialistAgent",
      "description": "Investigates fraud signals.",
      "instructions": "Investigate fraud indicators on the claim. Use lookup_enterprise_record and search_policy to assess the fraud signal and required steps.",
      "mcp_tools": [
        "lookup_enterprise_record",
        "search_policy"
      ],
      "mcp_server": "enterprise_context",
      "route_keywords": [
        "fraud",
        "signal",
        "mismatched",
        "investigation"
      ],
      "a2a_url": null
    },
    {
      "name": "ComplianceSpecialistAgent",
      "description": "Checks compliance holds.",
      "instructions": "Determine whether a compliance hold applies. Use search_policy to ground the claim exception routing policy.",
      "mcp_tools": [
        "search_policy"
      ],
      "mcp_server": "enterprise_context",
      "route_keywords": [
        "compliance",
        "hold",
        "regulatory"
      ],
      "a2a_url": null
    },
    {
      "name": "CustomerCommsAgent",
      "description": "Drafts the customer communication.",
      "instructions": "You receive the routed specialist's decision. Draft a clear, empathetic customer message for it. Use create_decision_log_entry to record the communicated outcome.",
      "mcp_tools": [
        "create_decision_log_entry"
      ],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        route_keywords=tuple(item.get("route_keywords", [])),
        a2a_url=item.get("a2a_url"),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_input=SCENARIO_DATA["sample_input"],
    agents=AGENTS,
    max_tokens=SCENARIO_DATA["max_tokens"],
    handoff_finisher=SCENARIO_DATA.get("handoff_finisher"),
    concurrent_synthesizer=SCENARIO_DATA.get("concurrent_synthesizer"),
    termination_phrases=tuple(SCENARIO_DATA.get("termination_phrases", [])),
)

print(f"Loaded {SCENARIO.title} with {len(SCENARIO.agents)} agents.")


### Inspection helpers

**Supporting code.** `agent_capability_map` summarizes who can do what, `mcp_tool_context` reports which domain tools exist, and `tools_for_agent` resolves an agent's granted tool names to the actual callables `make_agent` will attach. Inspecting the roster this way before running is a habit worth keeping: most orchestration surprises trace back to an agent having more, fewer, or different tools than you assumed.


In [ ]:
def tools_for_agent(spec: AgentSpec) -> list[object]:
    tools: list[object] = []
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "max_tokens": str(scenario.max_tokens),
        "sample": getattr(scenario, "sample_input"),
    }


def agent_capability_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "description": spec.description,
            "instructions": spec.instructions,
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }


### Sample prompt and budget

**Supporting code.** Pins the two run-defining inputs: `MAX_TOKENS`, the per-turn generation budget (this scenario's recommended value unless `OLLAMA_MAX_TOKENS` overrides it), and `SAMPLE_PROMPT`, the exact text the live run will dispatch. It then renders the roster so you can see the team -- and each agent's accent color -- before any orchestration happens. Budgets matter locally: too low truncates an agent mid- thought, too high slows every turn.


In [ ]:
import json


MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", str(SCENARIO.max_tokens)))
RESPONSES_PAYLOAD = {"input": SCENARIO.sample_input, "stream": False}
SAMPLE_PROMPT = SCENARIO.sample_input

render_roster(SCENARIO)
print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(agent_capability_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


### Ollama configuration

**Supporting code.** A frozen `OllamaAgentConfig` dataclass captures everything one agent's chat client needs -- model, host, temperature, context window, the scenario's token budget, keep-alive, and the think flag -- with environment variables as the override channel. Freezing it guarantees every agent in a run shares identical runtime settings. Local-runtime plumbing, independent of any Agent Framework class.


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.0
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

# Ollama's chat endpoint rejects a few OpenAI-style options; we strip these later.
_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "gemma4:12b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "1000")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


### Chat-client shim

**Supporting code.** A thin `OllamaChatClient` subclass that strips chat options the local Ollama server would reject before each request goes out. Adapters like this are common at the edge between a framework and a specific model server: the framework speaks a superset, the server accepts a subset, and the shim reconciles them without touching any agent code.


In [ ]:
class ScenarioOllamaChatClient(OllamaChatClient):
    """OllamaChatClient that drops chat options the local Ollama server rejects."""

    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


### make_agent

**Agent Framework primitive.** The factory this whole repo pivots on: `client.as_agent(...)` combines a chat client, the spec's role instructions (prefixed with the agent's name), and any granted tools into a runnable agent -- or returns an `A2AAgent` when the spec points at a remote peer. Every orchestration pattern downstream consumes agents built exactly here, which is why instructions and tool grants live in the scenario spec rather than in pattern code. This is the Agent Framework's core agent-construction call.


In [ ]:
def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    if spec.a2a_url:
        from agent_framework.a2a import A2AAgent

        url = spec.a2a_url
        if not url.startswith("http"):
            url = os.getenv("A2A_PARTNER_BASE_URL", "http://localhost:8765").rstrip("/") + url
        return A2AAgent(name=spec.name, description=spec.description, url=url)

    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


print("Agent factory ready: make_agent(spec) creates an instruction-led Ollama agent "
      "with its granted tools attached.")


### Framework imports and message helpers

**Agent Framework primitive.** Imports the workflow surface every pattern in this repo builds on -- `Executor`, `WorkflowBuilder`, `WorkflowContext`, `AgentExecutor`, and the `@handler` decorator -- plus `make_request` and `response_text`, two helpers that wrap plain text into an `AgentExecutorRequest` and pull it back out of a response. Messages are the typed boundary between workflow nodes: everything an agent receives or returns passes through these shapes.


In [ ]:
import re
from typing import Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


# State keys shared across executors: the running transcript, and the stopwords
# the handoff router strips when it derives routing keywords from agent names.
_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


### Transcript state

**Supporting code.** A helper that appends a `[AgentName] text` line to the shared transcript stored in workflow state via `ctx.get_state`/`ctx.set_state`. Keeping the transcript in state -- rather than inside any single executor -- is what lets gates, routers, and output executors all read the same running history. Bookkeeping the executors reuse, not framework API itself.


In [ ]:
def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


### Your first executor

**Agent Framework primitive.** `PromptDispatchExecutor` is the minimal custom executor: subclass `Executor`, mark an async method with `@handler`, and the framework routes matching messages to it. This one seeds the prompt and an empty transcript into state, then `send_message`s the first request -- the entry node of every graph in this repo. The handler signature (input type plus `WorkflowContext[OutputType]`) is how the framework knows what a node consumes and emits.


In [ ]:
class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


### Agents as workflow nodes

**Agent Framework primitive.** `_agent_executor` wraps a factory-built agent in an `AgentExecutor`, giving it a graph id and making it addressable as a workflow node -- the bridge between the agent world (instructions, tools, chat client) and the workflow world (edges, handlers, state). The slugified id matters more than it looks: the handoff router targets specialists by exactly these ids.


In [ ]:
def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))


print("Workflow plumbing ready: dispatch executor, shared transcript state, and "
      "request/response helpers.")


### Parse the ROUTE directive

**Supporting code.** A regex and a slugifier that read the `ROUTE: <AgentName>` line the triage agent was instructed to end with. Notice how forgiving the pattern is -- case-insensitive, tolerant of spacing -- because models format directives inconsistently; and notice what it does not do: validate. Deciding whether the named route is allowed belongs to the router, not the parser.


In [ ]:
_ROUTE_DIRECTIVE = re.compile(r"route\s*:\s*([A-Za-z][A-Za-z0-9 _-]*)", re.IGNORECASE)


def _route_slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


### HandoffRouterExecutor: validate the model's choice

**Agent Framework primitive.** The heart of the pattern: the model suggests, code decides. The plain methods are testable without a workflow -- `directed` honors a valid ROUTE line, `decide` falls back to scoring each specialist's keywords against the triage text and records which mechanism won -- and the `@handler` commits the choice to state and `send_message`s the chosen specialist via `target_id`. That `target_id` argument is the framework primitive that makes dynamic routing possible: one node, many possible next hops.


In [ ]:
class HandoffRouterExecutor(Executor):
    def __init__(
        self,
        id: str,
        *,
        routes: dict[str, tuple[str, ...]],
        default_route: str,
        display_names: dict[str, str] | None = None,
    ) -> None:
        super().__init__(id=id)
        self._routes = routes
        self._default_route = default_route
        self._display_names = display_names or {}

    def directed(self, text: str) -> str | None:
        for match in reversed(_ROUTE_DIRECTIVE.findall(text)):
            slug = _route_slug(match)
            if slug in self._routes:
                return slug
        return None

    def decide(self, text: str) -> tuple[str, str]:
        directed = self.directed(text)
        if directed is not None:
            return directed, "model-directive"
        lowered = text.lower()
        best_route, best_hits = self._default_route, 0
        for route, keywords in self._routes.items():
            hits = sum(1 for keyword in keywords if keyword in lowered)
            if hits > best_hits:
                best_route, best_hits = route, hits
        return best_route, "keyword-score"

    def choose(self, text: str) -> str:
        return self.decide(text)[0]

    @handler
    async def route(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        triage_text = response_text(response)
        chosen, source = self.decide(triage_text)
        ctx.set_state("route", chosen)
        ctx.set_state("route_name", self._display_names.get(chosen, chosen))
        ctx.set_state("route_source", source)
        prompt = ctx.get_state("prompt") or ""
        await ctx.send_message(
            make_request(f"Triage routed this to you.\nRequest:\n{prompt}\n\nTriage notes:\n{triage_text}"),
            target_id=chosen,
        )


### HandoffFinisherGateExecutor: hand off to a fixed finisher

**Agent Framework primitive.** When the scenario declares a finisher, every routed specialist's output flows through this gate to that fixed owner -- guaranteeing the run ends with the same accountable closing step (a customer letter, a final packet) no matter which route was taken. Route variance in the middle, an invariant ending: a very common compliance shape.


In [ ]:
class HandoffFinisherGateExecutor(Executor):
    @handler
    async def gate(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        route = ctx.get_state("route_name") or ctx.get_state("route") or "specialist"
        transcript = _append_transcript(ctx, route, response_text(response))
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(transcript)
        await ctx.send_message(
            make_request(
                f"You are the finishing stage of a handoff.\nOriginal request:\n{prompt}\n\n"
                f"Routed specialist notes:\n{carried}\n\nComplete the final deliverable."
            )
        )


### HandoffOutputExecutor: yield with a route header

**Agent Framework primitive.** The terminal `Executor` for handoff runs: it yields the answer prefixed with a `[routed to X via Y]` header, where Y is `model-directive` or `keyword-score`. That header is deliberately part of the output -- when you evaluate a handoff system, how often the model's routing was usable (versus rescued by the keyword fallback) is a metric you want visible on every run.


In [ ]:
class HandoffOutputExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str | None = None) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        route = ctx.get_state("route_name") or ctx.get_state("route") or "specialist"
        source = ctx.get_state("route_source") or "keyword-score"
        header = f"[routed to {route} via {source}]"
        if self._stage_name is None:
            await ctx.yield_output(f"{header}\n{response_text(response)}")
            return
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        await ctx.yield_output("\n\n".join([header, *transcript]))


### Derive routing keywords

**Supporting code.** Builds each specialist's keyword list -- the explicit `route_keywords` when the spec declares them, otherwise tokens derived from the agent's name and description with stopwords removed. This fallback vocabulary is the router's safety net when the model emits no usable directive, so skim it: if two specialists share their strongest keywords, ambiguous inputs will route unpredictably.


In [ ]:
def _route_keywords(spec: AgentSpec) -> tuple[str, ...]:
    if spec.route_keywords:
        return tuple(spec.route_keywords)
    tokens = re.findall(r"[a-z]+", f"{spec.name} {spec.description}".lower())
    keywords = [token for token in tokens if len(token) > 3 and token not in _STOPWORDS]
    return tuple(dict.fromkeys(keywords))[:6]


### Preview routing

**Supporting code.** An offline check -- no model call -- that exercises the router both ways: a well- formed ROUTE directive winning, then the keyword fallback scoring the sample prompt. Rerun it after editing any instruction or keyword; it is the fastest way to catch a routing regression before spending a live run on it.


In [ ]:
# Demo (offline): a valid ROUTE directive wins; keyword scoring is the fallback.
_specialists = [spec for spec in SCENARIO.agents[1:] if spec.name != SCENARIO.handoff_finisher]
_demo_routes = {_route_slug(spec.name): _route_keywords(spec) for spec in _specialists}
_demo_names = {_route_slug(spec.name): spec.name for spec in _specialists}
_demo_router = HandoffRouterExecutor(
    id="demo_router", routes=_demo_routes, default_route=next(iter(_demo_routes)), display_names=_demo_names
)
print("directive ->", _demo_router.choose("Triage notes.\nROUTE: " + _specialists[-1].name))
print("keywords  ->", _demo_router.choose(SAMPLE_PROMPT))


### Wire triage, router, and specialists with WorkflowBuilder

**Agent Framework primitive.** The graph runs dispatch -> triage -> router, then `add_edge`s the router to every specialist -- but at runtime the router's `target_id` picks exactly one of those edges to use. Compare the two closing shapes: without a finisher each specialist connects straight to output; with one, every specialist funnels through the finisher gate to the fixed owner. Edges define what is possible; the router decides what happens.


In [ ]:
def build_handoff_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    triage = _agent_executor(0, scenario, config=config)
    finisher_name = scenario.handoff_finisher
    routable = [i for i in range(1, len(scenario.agents)) if scenario.agents[i].name != finisher_name]
    specialists = [_agent_executor(i, scenario, config=config) for i in routable]
    specialist_ids = [_slug(scenario.agents[i].name) for i in routable]
    routes = {specialist_ids[pos]: _route_keywords(scenario.agents[i]) for pos, i in enumerate(routable)}
    display_names = {specialist_ids[pos]: scenario.agents[i].name for pos, i in enumerate(routable)}
    dispatch = PromptDispatchExecutor(id="dispatch")
    router = HandoffRouterExecutor(
        id="router", routes=routes, default_route=specialist_ids[0], display_names=display_names
    )
    output = HandoffOutputExecutor(id="final_output", stage_name=finisher_name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_edge(dispatch, triage)
    builder.add_edge(triage, router)
    if finisher_name is None:
        for specialist in specialists:
            builder.add_edge(router, specialist)
            builder.add_edge(specialist, output)
        return builder.build()
    finisher_index = next(
        i for i in range(1, len(scenario.agents)) if scenario.agents[i].name == finisher_name
    )
    finisher = _agent_executor(finisher_index, scenario, config=config)
    finisher_gate = HandoffFinisherGateExecutor(id="finisher_gate")
    for specialist in specialists:
        builder.add_edge(router, specialist)
        builder.add_edge(specialist, finisher_gate)
    builder.add_edge(finisher_gate, finisher)
    builder.add_edge(finisher, output)
    return builder.build()


### Compile and build

**Supporting code.** `build_workflow` resolves the Ollama config (model, host, and this scenario's token budget) and hands it to the builder above; `build()` then compiles the executor graph into a runnable workflow object. The wrapper is notebook glue so later cells can rebuild with overrides -- try `build_workflow(max_tokens=250)` for a faster smoke run -- while `build()` is the actual framework call. The printed type confirms what the framework produced.


In [ ]:
def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    return build_handoff_workflow(scenario, config=config)


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
print(
    f"Built the {SCENARIO.pattern} workflow over {len(SCENARIO.agents)} agents: "
    + type(workflow).__name__
)


## Flow Diagram

The diagram below shows a triage node routing to one of 4 specialists via keyword scoring attached to the Responses API /responses.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
Domain tool nodes use a stadium shape.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _handoff_diagram(scenario, api_boundary="Responses API /responses", input_label="OpenAI-style input")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram



def _handoff_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    triage, *others = scenario.agents
    finisher = next((agent for agent in others if agent.name == scenario.handoff_finisher), None)
    specialists = [agent for agent in others if agent is not finisher]
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append(f"    orchestrator --> triage[{_label(triage.name)}]")
    lines.append("    triage --> decision{Ownership decision}")
    pairs: list[tuple[AgentSpec, str]] = [(triage, "triage")]
    sink = "output[Responses output]"
    if finisher is not None:
        lines.append(f"    finisher[{_label(finisher.name)}] --> output[Responses output]")
        pairs.append((finisher, "finisher"))
        sink = "finisher"
    for index, agent in enumerate(specialists, start=1):
        node = f"specialist{index}"
        lines.append(f"    decision -->|handoff| {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> {sink}")
        pairs.append((agent, node))
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)



def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Responses API /responses")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
print(flow_diagram.mermaid)


### Read the run output

**Supporting code.** Utilities that unpack a finished run: `result.get_outputs()` returns the workflow's yielded outputs, and `get_intermediate_outputs()` exposes per-participant turns where the orchestration surfaces them (group chat and magentic). Everything else is string parsing that feeds `render_transcript`, so the color-coded turns you see below are exactly what the executors yielded -- those two calls are the only Agent Framework touchpoints.


In [ ]:
from collections.abc import Mapping, Sequence


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return clean_workflow_text(intermediate_text) or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return clean_workflow_text(intermediate_text)
    return clean_workflow_text(output_text) or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = (
        "termination condition",
        "maximum reset count",
        "maximum stall count",
        "workflow terminated",
        "group chat has reached its termination condition",
    )
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = clean_workflow_text(extract_text(value))
    return text


def clean_workflow_text(text: str) -> str:
    """Remove leading framework status lines when useful scenario text follows."""

    lines = text.splitlines()
    while lines and is_framework_status_line(lines[0]) and any(line.strip() for line in lines[1:]):
        lines.pop(0)
        while lines and not lines[0].strip():
            lines.pop(0)
    return "\n".join(lines).strip()


def is_framework_status_line(line: str) -> bool:
    normalized = line.strip().lower()
    return (
        normalized.startswith("invalid next speaker:")
        or normalized.startswith("magentic orchestrator:")
        or normalized.startswith("maximum consecutive function call errors reached")
    )


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")




print("Result formatting ready: workflow_result_to_text(...) turns framework events "
      "into readable text.")


## Live Run

Triage runs first and ends with a `ROUTE: <AgentName>` line. The `HandoffRouterExecutor` honors that directive when it names an allowed route, otherwise it scores each specialist keyword list against the triage text. If the scenario declares a finisher, the routed specialist's notes flow to that fixed agent, which completes the deliverable. The output shows the route taken and whether it came from the model directive or keyword fallback.

> **Instructor note:** `gemma4:12b` runs with `think: False` by default in this repo.
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
_framework_logs = io.StringIO()
with redirect_stdout(_framework_logs), redirect_stderr(_framework_logs):
    result = await workflow.run(SAMPLE_PROMPT)
framework_logs = _framework_logs.getvalue()
output_text = workflow_result_to_text(result)
if SCENARIO.pattern == "group-chat" and should_use_intermediate_outputs(output_text):
    output_text = group_chat_learning_summary(SCENARIO, SAMPLE_PROMPT, output_text)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

render_transcript(output_text)


## What to Inspect

Verify the route matches the triage intent, and check the route source in the output header: 'model-directive' means the triage agent's ROUTE line was honored; 'keyword-score' means the router fell back to scoring keywords. Try rewording the input toward a different specialist domain and rerun -- the route should change. Inspect `ctx.get_state('route')` and `ctx.get_state('route_source')` in the workflow state.

> **Scenario spotlight:** CLAIM-88120 has one fraud signal, so POL-CLM-09 routes fraud-first even though the amount also exceeds auto-approval -- check the ROUTE line honors that.


## Experiments

- Route CLAIM-88121 instead: it carries both a fraud signal and a compliance hold, so the fraud-first rule and the hold compete.
- Change `RESPONSES_PAYLOAD['input']` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `agent_capability_map(SCENARIO)` and tighten one agent's instructions to see how orchestration behavior changes.
- Lower `MAX_TOKENS` for faster smoke tests or raise it when handoff needs more room.
